In [1]:
#verifying whether properly connected to ADLS or not

for f in mssparkutils.fs.ls("Files/bronze"):
    print(f.name, f.isDir, f.size, f.path)

StatementMeta(, fa085f4d-9bc4-47d4-92a0-ffac9ea768a3, 3, Finished, Available, Finished, False)

customer True 0 abfss://eeacc14c-747c-48ae-ba46-b0c2ad355cd0@onelake.dfs.fabric.microsoft.com/c3a07f33-2a8d-4f58-abd6-6a463549b2f2/Files/bronze/customer
f_sls_t True 0 abfss://eeacc14c-747c-48ae-ba46-b0c2ad355cd0@onelake.dfs.fabric.microsoft.com/c3a07f33-2a8d-4f58-abd6-6a463549b2f2/Files/bronze/f_sls_t
product True 0 abfss://eeacc14c-747c-48ae-ba46-b0c2ad355cd0@onelake.dfs.fabric.microsoft.com/c3a07f33-2a8d-4f58-abd6-6a463549b2f2/Files/bronze/product
store True 0 abfss://eeacc14c-747c-48ae-ba46-b0c2ad355cd0@onelake.dfs.fabric.microsoft.com/c3a07f33-2a8d-4f58-abd6-6a463549b2f2/Files/bronze/store


In [2]:
#visualing sample data 

df_customer = spark.read.parquet("Files/bronze/customer")
df_product  = spark.read.parquet("Files/bronze/product")
df_store    = spark.read.parquet("Files/bronze/store")

for name, df in [("customer", df_customer), ("product", df_product), ("store", df_store)]:
    print(f"\n=== {name} ===")
    df.printSchema()
    df.show(5, truncate=False)
    print("Row count:", df.count())

StatementMeta(, fa085f4d-9bc4-47d4-92a0-ffac9ea768a3, 4, Finished, Available, Finished, False)


=== customer ===
root
 |-- cust_num: string (nullable = true)
 |-- cust_name: string (nullable = true)
 |-- phone: string (nullable = true)

+--------+--------------+--------------------+
|cust_num|cust_name     |phone               |
+--------+--------------+--------------------+
|C001    |Michael Davis |+1-210-343-3218x1960|
|C002    |Michael Miller|538.990.8386        |
|C003    |Carol Hays    |001-740-326-5423    |
|C004    |Joseph Ward   |361-855-9407        |
|C005    |Jamie Salinas |584-695-9310        |
+--------+--------------+--------------------+
only showing top 5 rows

Row count: 200

=== product ===
root
 |-- prod_num: string (nullable = true)
 |-- prod_name: string (nullable = true)

+--------+-------------------+
|prod_num|prod_name          |
+--------+-------------------+
|P001    |Like Camera        |
|P002    |Audience Television|
|P003    |Here Footwear      |
|P004    |Four Accessories   |
|P005    |Knowledge Bags     |
+--------+-------------------+
only showing

In [3]:
#transforming dimension data 

#FOR PRODUCT DIMENSION TABLE 

from pyspark.sql.functions import (
    col , trim , initcap , split , size , when , length , regexp_replace , 
    regexp_extract, row_number , lit , upper, current_timestamp
)

from pyspark.sql.window import Window

df_product_silver = (
    df_product
    .withColumn("prod_num" , upper(trim(col("prod_num"))))
    .withColumn("prod_name" , trim(initcap(col("prod_name"))))
    .filter(
        col("prod_num").isNotNull() & (col("prod_num") != "") &
        col("prod_name").isNotNull() & (col("prod_name") != "")
        
        )
    .orderBy("prod_num")
    .dropDuplicates(["prod_num"])
)


df_product_silver = df_product_silver.withColumn("silver_loaded_at", current_timestamp())


#save as delta
df_product_silver.write.format("delta").mode("overwrite").saveAsTable("silver_product")



StatementMeta(, fa085f4d-9bc4-47d4-92a0-ffac9ea768a3, 5, Finished, Available, Finished, False)

In [4]:
%%sql
--print our the prodcut table 

SELECT * FROM dbo.silver_product

LIMIT 5 ;

StatementMeta(, fa085f4d-9bc4-47d4-92a0-ffac9ea768a3, 6, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 3 fields>

In [5]:
#transforming next dimension table : Customer

#trim and uppercase Cust_num

from pyspark.sql.functions import (
    col, trim, upper, initcap, split, size, when, length,
    regexp_replace, regexp_extract, row_number, lit, current_timestamp, max as spark_max
)

from pyspark.sql.window import Window
from delta.tables import DeltaTable

df_customer_silver = df_customer.withColumn("cust_num" , upper(trim(col("cust_num"))))


#Split the full name 

df_customer_silver = df_customer_silver.withColumn(
    "name_parts", 
    split(trim(col("cust_name")) , " ")
    )

df_customer_silver = (
    df_customer_silver
    .withColumn("first_name" , col("name_parts")[0])
    .withColumn(
        "last_name",
        when(size(col("name_parts"))>= 2 , col("name_parts")[size(col("name_parts")) - 1])
        .otherwise(lit(None))
    )
    .withColumn(
        "middle_name",
        when(size(col("name_parts")) == 3 , col("name_parts")[1])
        .otherwise(lit(None))
    )
    .withColumnRenamed("cust_name" , "full_name")
    .drop("name_parts")
)


# Standarizing phone numbers

df_customer_silver = (
    df_customer_silver
    .withColumn(
        "phone_digits",
        regexp_replace(regexp_extract(col("phone"), r"^[^xX]*" , 0), r"[^0-9]","")
    )
    .withColumn(
        "phone",
        when(length(col("phone_digits")) >= 10,
        col("phone_digits").substr(length(col("phone_digits")) - 9 , lit(10)))
        .when(length(col("phone_digits")) >= 7  , col("phone_digits"))
        .otherwise(lit(None))
        )
        .drop("phone_digits")
)


#filtering invalid records

df_customer_silver = df_customer_silver.filter(
    col("cust_num").isNotNull() & (col("cust_num") != "") &
    col("full_name").isNotNull()
)

#drop duplicates

df_customer_silver = df_customer_silver.orderBy("cust_num").dropDuplicates(["cust_num"])

#adding a time stamp

df_customer_silver = df_customer_silver.withColumn("silver_loaded_at", current_timestamp())



StatementMeta(, fa085f4d-9bc4-47d4-92a0-ffac9ea768a3, 7, Finished, Available, Finished, False)

In [8]:
#save as delta
df_customer_silver.write.format("delta").mode("overwrite").saveAsTable("silver_customer")

StatementMeta(, fa085f4d-9bc4-47d4-92a0-ffac9ea768a3, 10, Finished, Available, Finished, False)

In [9]:
spark.table("silver_customer").show(10, truncate=False)
print("Row count:", spark.table("silver_customer").count())
print("Null phones:", spark.table("silver_customer").filter(col("phone").isNull()).count())


StatementMeta(, fa085f4d-9bc4-47d4-92a0-ffac9ea768a3, 11, Finished, Available, Finished, False)

+--------+---------------+----------+----------+---------+-----------+--------------------------+
|cust_num|full_name      |phone     |first_name|last_name|middle_name|silver_loaded_at          |
+--------+---------------+----------+----------+---------+-----------+--------------------------+
|C001    |Michael Davis  |2103433218|Michael   |Davis    |NULL       |2026-08-12 09:56:41.396938|
|C002    |Michael Miller |5389908386|Michael   |Miller   |NULL       |2026-08-12 09:56:41.396938|
|C003    |Carol Hays     |7403265423|Carol     |Hays     |NULL       |2026-08-12 09:56:41.396938|
|C004    |Joseph Ward    |3618559407|Joseph    |Ward     |NULL       |2026-08-12 09:56:41.396938|
|C005    |Jamie Salinas  |5846959310|Jamie     |Salinas  |NULL       |2026-08-12 09:56:41.396938|
|C006    |Danny Moore    |7315647525|Danny     |Moore    |NULL       |2026-08-12 09:56:41.396938|
|C007    |Erin Walker    |2415928327|Erin      |Walker   |NULL       |2026-08-12 09:56:41.396938|
|C008    |Isaiah Wil

In [10]:
#Transforming Store Dimension Table

df_store_silver = df_store.withColumn("store_num" , upper(trim(col("store_num"))))

df_store_silver = df_store_silver.withColumn("store_name", trim(col("store_name")))

df_store_silver = df_store_silver.withColumn(
    "address_parts" , split(trim(col("store_address")) , ",")

)

df_store_silver = (
    df_store_silver
    .withColumn("city", trim(col("address_parts")[0]))
    .withColumn("region" , trim(col("address_parts")[1]))
    .withColumnRenamed("store_address" , "full_address")
    .drop("address_parts")
)

#filter invalid records

df_store_silver = df_store_silver.filter(
    col("store_num").isNotNull() & (col("store_num") != "") &
    col("store_name").isNotNull() & (col("store_name") != "") 
)

#drop duplicates

df_store_silver = df_store_silver.orderBy("store_num").dropDuplicates(["store_num"])


#audit timestamp

df_store_silver = df_store_silver.withColumn("silver_loaded_at" , current_timestamp())

# save as delta table
df_store_silver.write.format("delta").mode("overwrite").saveAsTable("silver_store")




StatementMeta(, fa085f4d-9bc4-47d4-92a0-ffac9ea768a3, 12, Finished, Available, Finished, False)

In [11]:
spark.table("silver_store").show(truncate=False)
print("Row count:", spark.table("silver_store").count())

StatementMeta(, fa085f4d-9bc4-47d4-92a0-ffac9ea768a3, 13, Finished, Available, Finished, False)

+---------+-----------------------+----------------------------+--------------+------------+--------------------------+
|store_num|store_name             |full_address                |city          |region      |silver_loaded_at          |
+---------+-----------------------+----------------------------+--------------+------------+--------------------------+
|S001     |MegaMart Jimenezborough|Jimenezborough, South Region|Jimenezborough|South Region|2026-08-12 09:57:09.124085|
|S002     |MegaMart Peckmouth     |Peckmouth, East Region      |Peckmouth     |East Region |2026-08-12 09:57:09.124085|
|S003     |MegaMart New Michele   |New Michele, West Region    |New Michele   |West Region |2026-08-12 09:57:09.124085|
|S004     |MegaMart Brianahaven   |Brianahaven, North Region   |Brianahaven   |North Region|2026-08-12 09:57:09.124085|
|S005     |MegaMart Johnmouth     |Johnmouth, East Region      |Johnmouth     |East Region |2026-08-12 09:57:09.124085|
+---------+-----------------------+-----